
# Project 1 — fastText + MLP

## Architecture 1: Word Embeddings + Classifier

This notebook implements the first required architecture
for abductive commonsense reasoning on the ART dataset.

Architecture:
- Frozen pretrained fastText embeddings (300d)
- Mean pooling for each of the four text segments
- Concatenation (1200d)
- Linear (1200 → 128)
- ReLU
- Linear (128 → 2)

Only the classifier is trained.


In [1]:

import sys
from pathlib import Path

import torch
import torch.nn as nn

# Add repository root to Python path
repo_root = Path.cwd().parents[1]

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

# Load saved preprocessing artifacts
artifact_path = repo_root / "project1" / "artifacts" / "fasttext_art.pt"

artifacts = torch.load(
    artifact_path,
    map_location="cpu",
    weights_only=True
)

vocab = artifacts["vocab"]
embedding_weights = artifacts["embedding_weights"]

# Restore frozen embedding layer
embedding_layer = nn.Embedding.from_pretrained(
    embedding_weights,
    freeze=True,
    padding_idx=vocab["[PAD]"]
)

print("Vocabulary size:", len(vocab))
print("Embedding shape:", embedding_layer.weight.shape)
print("Embeddings frozen:", not embedding_layer.weight.requires_grad)


Vocabulary size: 15009
Embedding shape: torch.Size([15009, 300])
Embeddings frozen: True


In [2]:
from datasets import load_dataset
from project1.src.data_loader import create_dataloader

# Load ART dataset
dataset = load_dataset("amarfurt/art-nlp-hs26")

# Create DataLoaders
batch_size = 32

train_loader = create_dataloader(
    dataset["train"],
    vocab,
    batch_size=batch_size,
    shuffle=True
)

val_loader = create_dataloader(
    dataset["validation"],
    vocab,
    batch_size=batch_size,
    shuffle=False
)

print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))


C:\Users\kimbo\anaconda3\envs\nlp-art-hs26\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Training batches: 526
Validation batches: 32


In [3]:

# Get one batch
batch = next(iter(train_loader))

input_ids = batch["input_ids"]
lengths = batch["lengths"]
labels = batch["labels"]

# Pass through frozen embedding layer
embeddings = embedding_layer(input_ids)

print("Input IDs:", input_ids.shape)
print("Embeddings:", embeddings.shape)
print("Labels:", labels.shape)

# Verify padding
print("\nSequence lengths correct:",
      torch.equal(
          lengths,
          (input_ids != vocab["[PAD]"]).sum(dim=1)
      ))

# Verify that each example has exactly 3 separators
sep_counts = (input_ids == vocab["[SEP]"]).sum(dim=1)

print("Each example has 3 separators:",
      torch.all(sep_counts == 3).item())


Input IDs: torch.Size([32, 60])
Embeddings: torch.Size([32, 60, 300])
Labels: torch.Size([32])

Sequence lengths correct: True
Each example has 3 separators: True


In [4]:

from project1.src.models import EmbeddingMLP

# Initialize model
model = EmbeddingMLP(
    embedding_layer=embedding_layer,
    pad_id=vocab["[PAD]"],
    sep_id=vocab["[SEP]"],
    hidden_dim=128
)

# Use our existing batch
batch = next(iter(train_loader))

# Forward pass
with torch.no_grad():
    logits = model(batch["input_ids"])

print("Input shape:", batch["input_ids"].shape)
print("Output shape:", logits.shape)

print("\nFirst 5 logits:")
print(logits[:5])

print("\nPredicted classes:")
print(logits.argmax(dim=1)[:5])

print("\nActual classes:")
print(batch["labels"][:5])

# Check trainable parameters
trainable_params = sum(
    p.numel() for p in model.parameters()
    if p.requires_grad
)

print("\nTrainable parameters:", trainable_params)
print("Embeddings frozen:", not model.embedding.weight.requires_grad)


Input shape: torch.Size([32, 57])
Output shape: torch.Size([32, 2])

First 5 logits:
tensor([[-0.0710,  0.0210],
        [-0.0809,  0.0200],
        [-0.0737,  0.0145],
        [-0.0755,  0.0205],
        [-0.0723,  0.0346]])

Predicted classes:
tensor([1, 1, 1, 1, 1])

Actual classes:
tensor([1, 1, 0, 1, 0])

Trainable parameters: 153986
Embeddings frozen: True


In [5]:
import wandb

# Initialize a test experiment
run = wandb.init(
    project="nlp-art-hs26-project1",
    name="setup-test",
    job_type="setup",
    config={
        "model": "fasttext-mlp",
        "embedding_dim": 300,
        "hidden_dim": 128,
        "batch_size": 32
    }
)

# Log one test metric
run.log({"test/connection": 1})

print("Project:", run.project)
print("Entity:", run.entity)
print("Run URL:", run.url)

# Finish the experiment
run.finish()


wandb: ERROR Failed to detect the name of this notebook. You can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: [wandb.login()] Loaded credentials for https://api.forge.coreweave.com from C:\Users\kimbo\_netrc.
wandb: Currently logged in as: kimbortis (kimbortis-hochschule-luzern) to https://api.forge.coreweave.com. Use `wandb login --relogin` to force relogin


Project: nlp-art-hs26-project1
Entity: kimbortis-hochschule-luzern
Run URL: https://forge.coreweave.com/wandb/kimbortis-hochschule-luzern/nlp-art-hs26-project1/runs/v4favyi1


test/connection,▁
test/connection,1


In [6]:
import torch.nn as nn
from project1.src.training import evaluate

# Our loss function
criterion = nn.CrossEntropyLoss()

# Use CPU for now
device = torch.device("cpu")
model = model.to(device)

# Evaluate the untrained model
val_metrics = evaluate(
    model,
    val_loader,
    criterion,
    device
)

print(f"Validation loss: {val_metrics['loss']:.4f}")
print(f"Validation accuracy: {val_metrics['accuracy']:.2%}")


Validation loss: 0.6951
Validation accuracy: 49.20%


In [7]:
import random
import numpy as np
import torch
import torch.nn as nn

from project1.src.models import EmbeddingMLP
from project1.src.training import train_one_epoch, evaluate

# Reproducibility
seed = 42

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

# Hyperparameters
config = {
    "architecture": "fasttext-mlp",
    "embedding_dim": 300,
    "hidden_dim": 128,
    "batch_size": 32,
    "learning_rate": 0.001,
    "optimizer": "Adam",
    "max_epochs": 15,
    "patience": 4,
    "seed": seed
}

# Device
device = torch.device("cpu")

# Initialize a new model
model = EmbeddingMLP(
    embedding_layer=embedding_layer,
    pad_id=vocab["[PAD]"],
    sep_id=vocab["[SEP]"],
    hidden_dim=config["hidden_dim"]
).to(device)

# Loss and optimizer
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=config["learning_rate"]
)

print("Device:", device)
print("Optimizer:", optimizer.__class__.__name__)
print("Learning rate:", config["learning_rate"])
print("Training examples:", len(train_loader.dataset))
print("Validation examples:", len(val_loader.dataset))


Device: cpu
Optimizer: Adam
Learning rate: 0.001
Training examples: 16801
Validation examples: 1000


In [8]:

import wandb
from pathlib import Path

# Directory for model checkpoints
checkpoint_dir = repo_root / "project1" / "artifacts"
checkpoint_dir.mkdir(parents=True, exist_ok=True)

checkpoint_path = checkpoint_dir / "mlp_best.pt"

# Initialize W&B experiment
run = wandb.init(
    entity="kimbortis-hochschule-luzern",
    project="nlp-art-hs26-project1",
    group="fasttext-mlp",
    name="mlp-baseline-001",
    config=config
)

best_val_accuracy = -1.0
best_val_loss = float("inf")
epochs_without_improvement = 0

try:
    for epoch in range(1, config["max_epochs"] + 1):

        # Train
        train_metrics = train_one_epoch(
            model, train_loader, optimizer, criterion, device
        )

        # Validate
        val_metrics = evaluate(
            model, val_loader, criterion, device
        )

        # Track metrics
        run.log({
            "epoch": epoch,
            "train/loss": train_metrics["loss"],
            "train/accuracy": train_metrics["accuracy"],
            "val/loss": val_metrics["loss"],
            "val/accuracy": val_metrics["accuracy"]
        })

        print(
            f"Epoch {epoch:02d} | "
            f"Train loss: {train_metrics['loss']:.4f} | "
            f"Train acc: {train_metrics['accuracy']:.2%} | "
            f"Val loss: {val_metrics['loss']:.4f} | "
            f"Val acc: {val_metrics['accuracy']:.2%}"
        )

        # Select best checkpoint by validation accuracy,
        # breaking ties using validation loss
        improved = (
            val_metrics["accuracy"] > best_val_accuracy
            or (
                val_metrics["accuracy"] == best_val_accuracy
                and val_metrics["loss"] < best_val_loss - 1e-4
            )
        )

        if improved:
            best_val_accuracy = val_metrics["accuracy"]
            best_val_loss = val_metrics["loss"]
            epochs_without_improvement = 0

            torch.save(
                model.state_dict(),
                checkpoint_path
            )

            run.summary["best_epoch"] = epoch
            run.summary["best_val_accuracy"] = best_val_accuracy

        else:
            epochs_without_improvement += 1

        # Early stopping
        if epochs_without_improvement >= config["patience"]:
            print(f"\nEarly stopping at epoch {epoch}")
            break

    print("\nBest validation accuracy:",
          f"{best_val_accuracy:.2%}")
    print("Checkpoint:", checkpoint_path)

finally:
    run.finish()


Epoch 01 | Train loss: 0.6921 | Train acc: 51.68% | Val loss: 0.6891 | Val acc: 53.00%
Epoch 02 | Train loss: 0.6788 | Train acc: 57.04% | Val loss: 0.6754 | Val acc: 56.50%
Epoch 03 | Train loss: 0.6591 | Train acc: 61.23% | Val loss: 0.6638 | Val acc: 58.30%
Epoch 04 | Train loss: 0.6453 | Train acc: 62.51% | Val loss: 0.6555 | Val acc: 59.90%
Epoch 05 | Train loss: 0.6340 | Train acc: 63.72% | Val loss: 0.6559 | Val acc: 59.70%
Epoch 06 | Train loss: 0.6248 | Train acc: 64.99% | Val loss: 0.6484 | Val acc: 62.20%
Epoch 07 | Train loss: 0.6170 | Train acc: 65.64% | Val loss: 0.6480 | Val acc: 60.60%
Epoch 08 | Train loss: 0.6100 | Train acc: 66.79% | Val loss: 0.6500 | Val acc: 60.80%
Epoch 09 | Train loss: 0.6062 | Train acc: 66.65% | Val loss: 0.6545 | Val acc: 59.80%
Epoch 10 | Train loss: 0.6020 | Train acc: 66.91% | Val loss: 0.6454 | Val acc: 60.30%

Early stopping at epoch 10

Best validation accuracy: 62.20%
Checkpoint: C:\Users\kimbo\Documents\HSLU\Sem 5\NLP\nlp-art-hs26-kim

epoch,▁▂▃▃▄▅▆▆▇█
train/accuracy,▁▃▅▆▇▇▇███
train/loss,█▇▅▄▃▃▂▂▁▁
val/accuracy,▁▄▅▆▆█▇▇▆▇
val/loss,█▆▄▃▃▁▁▂▂▁
best_epoch,6
best_val_accuracy,0.622
epoch,10
train/accuracy,0.66913
train/loss,0.60198
val/accuracy,0.603


In [9]:

# Load the best model weights (epoch 6)
model.load_state_dict(
    torch.load(
        checkpoint_path,
        map_location=device,
        weights_only=True
    )
)

# Evaluate the restored model
best_metrics = evaluate(
    model,
    val_loader,
    criterion,
    device
)

print("Restored model results:")
print(f"Validation loss: {best_metrics['loss']:.4f}")
print(f"Validation accuracy: {best_metrics['accuracy']:.2%}")


Restored model results:
Validation loss: 0.6484
Validation accuracy: 62.20%
